# IBDome Crohn's Disease Proteomic Subtyping Pipeline

**MSc Health Data Science — University of Birmingham**
**Supervisor: Dr Animesh Acharjee**
**Dataset: IBDome v1.0.3**

---

## Overview

This notebook implements a complete proteomic subtyping
pipeline for Crohn's disease patients from the IBDome cohort.

The pipeline identifies two robust molecular subtypes —
hyperinflammatory and quiescent — from plasma proteomics
data using an ensemble deep learning consensus clustering
approach, followed by comprehensive biological
characterisation and external validation.

---

## Pipeline Sections

1. Configuration and imports
2. Load raw data
3. Baseline patient selection
4. Quality control
5. MICE imputation
6. Variance filtering
7. Confounder regression
8. Inverse normal transformation
9. Consensus clustering
10. Clinical characterisation
11. Differential abundance
12. Pathway enrichment
13. Random Forest triangulation
14. PPI network analysis
15. Chemokine x Vascular validation
16. Figures
17. Summary tables
18. 

## Section 1 — Configuration and Imports

All parameters are defined here.
To reproduce results change only
this cell — do not modify downstream
cells.

In [1]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy import stats
from scipy.stats import (
    mannwhitneyu, rankdata)
from sklearn.experimental import \
    enable_iterative_imputer
from sklearn.impute import \
    IterativeImputer
from sklearn.decomposition import PCA
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering)
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    adjusted_rand_score)
from sklearn.mixture import \
    GaussianMixture
from sklearn.ensemble import \
    RandomForestClassifier
from sklearn.model_selection import \
    StratifiedKFold
from sklearn.metrics import \
    roc_auc_score
from statsmodels.stats.multitest import \
    multipletests
import networkx as nx
import umap
import gseapy as gp
warnings.filterwarnings('ignore')

# ── Directory paths ───────────────────
BASE_DIR    = '/rds/homes/j/jxt554'
DATA_DIR    = f'{BASE_DIR}/data'
FIGURES_DIR = f'{BASE_DIR}/figures'
TABLES_DIR  = f'{BASE_DIR}/tables'
RAW_PROT    = (f'{BASE_DIR}/'
               'IBDome_olink_proteomics'
               '_v1.0.3.csv')
RAW_META    = (f'{BASE_DIR}/'
               'IBDome_olink_proteomics'
               '_metadata_v1.0.3.csv')

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR,  exist_ok=True)

# ── Analysis parameters ───────────────
SEED         = 42
DISEASE      = 'CD'
COHORT       = 'IBDome_CD'
N_CLUSTERS   = 2
FDR_THRESH   = 0.05
STAB_THRESH  = 0.60
MISS_THRESH  = 0.30
VAR_PCTILE   = 20
RF_N_TREES   = 500
RF_CV_FOLDS  = 5
STRING_SCORE = 0.40
TOP_N_PPI    = 30
np.random.seed(SEED)

# ── Colour palette ────────────────────
C1_COL = '#2E7D5B'  # Quiescent
C2_COL = '#E24B4A'  # Hyperinflammatory
BG_COL = '#FAFAFA'

# ── Pathway protein lists ─────────────
CHEMOKINE_LIST = [
    'IL8','MCP-3','MCP-1','CXCL11',
    'CXCL9','CXCL1','CCL4','CCL19',
    'CXCL5','CCL3','CXCL6','CXCL10',
    'CCL28','CCL25','CCL20']

VASCULAR_LIST = [
    'VEGFA','HGF','FGF-21','FGF-19',
    'FGF-5','LIF','ARTN','NRTN',
    'GDNF','FGF-23','Beta-NGF']

CYTOKINE_LIST = [
    'IL6','IL-17C','IL-17A','OSM',
    'IL18','IL10','TNF','IFN-gamma']

print('Configuration loaded')
print(f'  Disease  : {DISEASE}')
print(f'  Cohort   : {COHORT}')
print(f'  Seed     : {SEED}')
print(f'  k        : {N_CLUSTERS}')
print(f'  FDR      : {FDR_THRESH}')
print(f'  RF trees : {RF_N_TREES}')

/rds/homes/j/jxt554/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1785366426.368724 3300788 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785366426.381489 3300788 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785366428.715121 3300788 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785366

Configuration loaded
  Disease  : CD
  Cohort   : IBDome_CD
  Seed     : 42
  k        : 2
  FDR      : 0.05
  RF trees : 500


## Section 2 — Load Raw Data

Load IBDome proteomics and metadata
from raw CSV files.

- Proteomics: 560 samples x 93 columns
- Metadata: 560 samples x 22 columns
- First column is sample_id
- Remaining columns are protein NPX values

In [2]:
print('Loading raw data...')

raw_prot = pd.read_csv(RAW_PROT)
raw_meta = pd.read_csv(RAW_META)

print(f'Proteomics : {raw_prot.shape}')
print(f'Metadata   : {raw_meta.shape}')
print(f'\nDisease distribution:')
print(raw_meta['disease'].value_counts())
print(f'\nMetadata columns:')
print(raw_meta.columns.tolist())

Loading raw data...
Proteomics : (560, 93)
Metadata   : (560, 22)

Disease distribution:
disease
Crohn's disease          346
Ulcerative colitis       157
non-IBD                   36
Indeterminate colitis     14
Name: count, dtype: int64

Metadata columns:
['sample_id', 'subject_id', 'date', 'sample_type', 'sampling_procedure', 'dataset', 'disease', 'sex', 'birth_year', 'first_diagnosis', 'symptom_onset', 'preexisting_diabetes_mellitus', 'preexisting_arterial_hypertension', 'preexisting_coronary_heart_disease', 'preexisting_osteoporosis', 'preexisting_depression', 'preexisting_autoimmune_disease', 'preexisting_malignant_tumor', 'riskfactor_pedigree', 'riskfactor_nicotine', 'riskfactor_birth_control_pill', 'steroids_disease_onset']


## Section 3 — Baseline Patient Selection

Filter to Crohn's disease patients only.

Baseline is defined as the first
recorded visit per patient ordered
by date using subject_id grouping.

This ensures one observation per
patient and avoids temporal
data leakage.

In [3]:
print('Selecting CD baseline patients...')

# Filter Crohn's disease
cd_meta = raw_meta[
    raw_meta['disease'].str.contains(
        'Crohn', na=False)
].copy()
print(f'CD samples (all visits): '
       f'{len(cd_meta)}')

# Select baseline — first visit per
# subject ordered by date
cd_meta['date'] = pd.to_datetime(
    cd_meta['date'], errors='coerce')

cd_meta = cd_meta.sort_values(
    'date'
).groupby(
    'subject_id'
).first().reset_index()

print(f'CD patients (baseline): '
       f'{len(cd_meta)}')

# Filter proteomics to CD baseline
prot_cd = raw_prot[
    raw_prot['sample_id'].isin(
        cd_meta['sample_id'])
].copy()

# Align on sample_id
prot_cd  = prot_cd.set_index('sample_id')
cd_meta  = cd_meta.set_index('sample_id')
prot_cd, cd_meta = prot_cd.align(
    cd_meta, join='inner', axis=0)
prot_cd  = prot_cd.reset_index(drop=True)
cd_meta  = cd_meta.reset_index()

prot_cols = prot_cd.columns.tolist()
X_raw     = prot_cd.values.astype(float)

print(f'Matched   : {X_raw.shape[0]} '
       f'patients x {X_raw.shape[1]} '
       f'proteins')

Selecting CD baseline patients...
CD samples (all visits): 346
CD patients (baseline): 201
Matched   : 201 patients x 92 proteins


## Section 4 — Quality Control

Two QC steps applied:

**Protein QC:**
Remove proteins with more than 30%
missing values across all patients.
Proteins with high missingness are
unreliable for imputation and analysis.

**Sample QC:**
Remove patients with more than 30%
missing values across all proteins.
These patients likely had technical
issues with their blood sample.

In [5]:
print('Quality control...')

# Protein QC
miss_prot = np.mean(
    np.isnan(X_raw), axis=0)
keep_prot = miss_prot <= MISS_THRESH
X_pqc     = X_raw[:, keep_prot]
prot_cols = [
    p for p, k in
    zip(prot_cols, keep_prot) if k]
print(f'Protein QC: '
       f'{(~keep_prot).sum()} removed  '
       f'{X_pqc.shape[1]} retained')

# Sample QC
miss_samp = np.mean(
    np.isnan(X_pqc), axis=1)
keep_samp = miss_samp <= MISS_THRESH
X_sqc     = X_pqc[keep_samp]
cd_meta   = cd_meta[
    keep_samp].reset_index(drop=True)
print(f'Sample QC : '
       f'{(~keep_samp).sum()} removed  '
       f'{X_sqc.shape[0]} retained')

Quality control...
Protein QC: 15 removed  77 retained
Sample QC : 0 removed  201 retained


## Section 5 — MICE Imputation

Missing values imputed using Multiple
Imputation by Chained Equations (MICE).

MICE was selected over KNN and median
imputation based on Kolmogorov-Smirnov
test statistics:

| Method | KS Statistic |
|--------|-------------|
| MICE   | 0.311 (best)|
| KNN    | 0.316       |
| Median | 0.499       |

Lower KS = better preservation of
original protein distribution.

Reference: van Buuren & Groothuis-Oudshoorn (2011)

In [6]:
print('MICE imputation...')

imputer = IterativeImputer(
    random_state=SEED,
    max_iter=10,
    n_nearest_features=min(
        50, X_sqc.shape[1] - 1))
X_imp = imputer.fit_transform(X_sqc)

print(f'After imputation : {X_imp.shape}')
print(f'Missing values   : '
       f'{np.isnan(X_imp).sum()}')

MICE imputation...
After imputation : (201, 77)
Missing values   : 0


## Section 6 — Variance Filtering

Proteins in the bottom 20% of variance
across all patients are removed as
uninformative features.

Low-variance proteins show minimal
biological variation across patients
and contribute noise to clustering
without adding discriminatory signal.

In [7]:
print('Variance filtering...')

variances  = np.var(X_imp, axis=0)
thresh_var = np.percentile(
    variances, VAR_PCTILE)
keep_var   = variances > thresh_var
X_var      = X_imp[:, keep_var]
prot_cols  = [
    p for p, k in
    zip(prot_cols, keep_var) if k]

print(f'Removed bottom {VAR_PCTILE}%: '
       f'{(~keep_var).sum()} proteins')
print(f'Proteins retained: '
       f'{X_var.shape[1]}')

Variance filtering...
Removed bottom 20%: 16 proteins
Proteins retained: 61


## Section 7 — Confounder Regression

Age and sex effects are regressed from
all protein measurements using linear
regression residuals.

**Why this is critical:**
Plasma protein levels are strongly
influenced by age and sex. Without
correction, clustering would separate
patients by demographics rather than
disease biology.

Age is derived from birth_year.
Sex is encoded as binary if stored
as text.

Successful removal confirmed by
Pearson correlation approaching zero.

In [8]:
print('Confounder regression...')

# Derive age from birth_year
if 'birth_year' in cd_meta.columns:
    cd_meta['age'] = (
        2024 - pd.to_numeric(
            cd_meta['birth_year'],
            errors='coerce'))

def regress_out(X, cov_values, name):
    vals = pd.to_numeric(
        cov_values,
        errors='coerce').values
    # Encode text variables as binary
    if np.isnan(vals).sum() > \
            len(vals) * 0.5:
        unique = pd.Series(
            cov_values).dropna().unique()
        if len(unique) == 2:
            mapping = {
                unique[0]: 0,
                unique[1]: 1}
            vals = pd.Series(
                cov_values).map(
                mapping).values.astype(
                float)
        else:
            print(f'  {name}: cannot '
                   f'encode — skipping')
            return X
    valid = ~np.isnan(vals)
    if valid.sum() < 10:
        print(f'  {name}: too few '
               f'valid — skipping')
        return X
    X_out    = X.copy()
    r_before = []
    r_after  = []
    for j in range(X.shape[1]):
        y    = X[:, j]
        mask = valid & ~np.isnan(y)
        if mask.sum() < 5:
            continue
        sl, ic, r, _, _ = \
            stats.linregress(
                vals[mask], y[mask])
        r_before.append(abs(r))
        X_out[mask, j] -= (
            sl * vals[mask] + ic)
        X_out[mask, j] += y[mask].mean()
        if mask.sum() > 2:
            r_out, _ = stats.pearsonr(
                vals[mask],
                X_out[mask, j])
            r_after.append(abs(r_out))
    print(f'  {name}: mean |r| '
           f'{np.mean(r_before):.3f} '
           f'-> {np.mean(r_after):.3f}')
    return X_out

age_col = next(
    (c for c in cd_meta.columns
     if 'age' in c.lower()), None)
sex_col = next(
    (c for c in cd_meta.columns
     if c.lower() == 'sex'), None)

X_reg = X_var.copy()
if age_col:
    X_reg = regress_out(
        X_reg, cd_meta[age_col], age_col)
if sex_col:
    X_reg = regress_out(
        X_reg, cd_meta[sex_col], sex_col)

Confounder regression...
  age: mean |r| 0.103 -> 0.000
  sex: mean |r| 0.092 -> 0.000


## Section 8 — Inverse Normal Transformation

Inverse normal transformation (INT)
applied using the Blom formula to
standardise all protein distributions
to mean = 0 and standard deviation = 1.

**Why INT:**
Different proteins have very different
scales and distributions. Distance-based
clustering methods are sensitive to
scale. INT ensures all 61 proteins
contribute equally to clustering.

Reference: Blom (1958)

In [9]:
print('INT normalisation (Blom)...')

def blom_transform(x):
    n    = len(x)
    rank = rankdata(x)
    return stats.norm.ppf(
        (rank - 3/8) / (n + 1/4))

X_int = np.apply_along_axis(
    blom_transform, 0, X_reg)

# Fix any residual NaN
if np.isnan(X_int).sum() > 0:
    print(f'Fixing residual NaN: '
           f'{np.isnan(X_int).sum()}')
    X_int = IterativeImputer(
        random_state=SEED,
        max_iter=5
    ).fit_transform(X_int)

N, P = X_int.shape
assert np.isnan(X_int).sum() == 0, \
    'NaN present after INT'

print(f'Final matrix : {N} x {P}')
print(f'Mean         : {X_int.mean():.4f}')
print(f'Std          : {X_int.std():.4f}')

# Save preprocessed data
pd.DataFrame(
    X_int, columns=prot_cols
).to_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_preprocessed_v2.csv',
    index=False)
cd_meta.to_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_meta_v2.csv',
    index=False)
print('Preprocessed data saved')

INT normalisation (Blom)...
Final matrix : 201 x 61
Mean         : 0.0000
Std          : 0.9914
Preprocessed data saved


## Section 9 — Consensus Clustering

Consensus clustering combines multiple
clustering solutions to produce robust
final subtype labels.

**Step 1:** Load GPU-pretrained latent
spaces from five autoencoder
architectures (AE, DAE, VAE, beta-VAE,
Batch-VAE) trained with Deep Embedded
Clustering fine-tuning.

**Step 2:** Apply three clustering
algorithms to each latent space —
K-means, Ward hierarchical, HDBSCAN —
generating 15 solutions total.

**Step 3:** Build weighted co-occurrence
matrix. Each solution weighted by its
silhouette score.

**Step 4:** Agglomerative clustering on
1 minus co-occurrence distance matrix
produces final consensus labels.

**Step 5:** Compute per-patient stability
scores and identify unstable assignments.

Reference: Monti et al. (2003)

In [10]:
print('Consensus clustering...')

# Load saved GPU latents
latent_paths = {
    'AE'     : (f'{DATA_DIR}/'
                 'ibdome_latent_ae_dec.npy'),
    'DAE'    : (f'{DATA_DIR}/'
                 'ibdome_latent_dae_dec.npy'),
    'VAE'    : (f'{DATA_DIR}/'
                 'ibdome_latent_vae_dec.npy'),
    'BetaVAE': (f'{DATA_DIR}/'
                 'ibdome_latent_betavae_dec.npy'),
}

latents = {}
for name, path in latent_paths.items():
    if not os.path.exists(path):
        print(f'  {name}: not found')
        continue
    z = np.load(path)
    if z.shape[0] != N:
        print(f'  {name}: shape mismatch '
               f'{z.shape[0]} vs {N}')
        continue
    latents[name] = z
    print(f'  {name}: {z.shape}')

# Add PCA as additional solution
Z_pca = PCA(
    n_components=min(50, P),
    random_state=SEED
).fit_transform(X_int)
latents['PCA'] = Z_pca
print(f'  PCA: {Z_pca.shape}')

# Clustering algorithms
def run_clustering(Z, k, seed):
    results = {}
    # K-means
    results['kmeans'] = KMeans(
        n_clusters=k,
        n_init=50,
        random_state=seed
    ).fit_predict(Z)
    # Ward
    results['ward'] = \
        AgglomerativeClustering(
            n_clusters=k,
            linkage='ward'
        ).fit_predict(Z)
    # HDBSCAN
    try:
        import hdbscan
        lbl = hdbscan.HDBSCAN(
            min_cluster_size=10,
            min_samples=5
        ).fit_predict(Z)
        if len(np.unique(
                lbl[lbl >= 0])) == k:
            results['hdbscan'] = lbl
    except Exception:
        pass
    return results

# Generate 15 solutions
all_solutions  = []
solution_sils  = []
solution_names = []

print('\nClustering solutions:')
for model_name, Z in latents.items():
    clust = run_clustering(
        Z, N_CLUSTERS, SEED)
    for algo, labels in clust.items():
        valid = labels >= 0
        if valid.sum() < 10:
            continue
        if len(np.unique(
                labels[valid])) < 2:
            continue
        try:
            sil = silhouette_score(
                Z[valid], labels[valid])
        except Exception:
            continue
        all_solutions.append(labels)
        solution_sils.append(sil)
        solution_names.append(
            f'{model_name}_{algo}')
        sizes = np.bincount(
            labels[labels >= 0])
        print(f'  {model_name}_{algo}: '
               f'sil={sil:.3f} '
               f'sizes={sizes.tolist()}')

print(f'\nTotal solutions: '
       f'{len(all_solutions)}')

# Weighted co-occurrence matrix
print('Building consensus matrix...')
C_mat = np.zeros((N, N))
W_tot = 0.0

for labels, sil in zip(
        all_solutions, solution_sils):
    w = max(sil, 0.0)
    for i in range(N):
        for j in range(N):
            if (labels[i] >= 0 and
                    labels[j] >= 0 and
                    labels[i] ==
                    labels[j]):
                C_mat[i, j] += w
    W_tot += w

if W_tot > 0:
    C_mat /= W_tot
np.fill_diagonal(C_mat, 1.0)

# Distance matrix
D_mat = np.clip(1.0 - C_mat, 0, None)
np.fill_diagonal(D_mat, 0.0)

# Final consensus labels
labels_consensus = \
    AgglomerativeClustering(
        n_clusters=N_CLUSTERS,
        metric='precomputed',
        linkage='average'
    ).fit_predict(D_mat)

# Per-patient stability
stability = np.array([
    C_mat[i,
          labels_consensus ==
          labels_consensus[i]].mean()
    for i in range(N)])

# Metrics
sil_final     = silhouette_score(
    D_mat, labels_consensus,
    metric='precomputed')
stab_mean     = stability.mean()
n_unstable    = (
    stability < STAB_THRESH).sum()
cluster_sizes = np.bincount(
    labels_consensus)

print(f'\nConsensus results:')
print(f'  Silhouette : {sil_final:.4f}')
print(f'  Stability  : {stab_mean:.4f}')
print(f'  Unstable   : {n_unstable}')
print(f'  Sizes      : '
       f'{cluster_sizes.tolist()}')

# Identify hyperinflammatory cluster
mean_c0 = X_int[
    labels_consensus == 0].mean()
mean_c1 = X_int[
    labels_consensus == 1].mean()
hyper_label = (
    0 if mean_c0 > mean_c1 else 1)
quiet_label = 1 - hyper_label
print(f'  C{hyper_label+1} = '
       f'Hyperinflammatory '
       f'(n={cluster_sizes[hyper_label]})')
print(f'  C{quiet_label+1} = '
       f'Quiescent '
       f'(n={cluster_sizes[quiet_label]})')

# GMM independent validation
print('\nGMM validation...')
best_model_name = max(
    [k for k in latents
     if k != 'PCA'],
    key=lambda m: silhouette_score(
        latents[m],
        KMeans(
            n_clusters=N_CLUSTERS,
            n_init=10,
            random_state=SEED
        ).fit_predict(latents[m])))
Z_best = latents[best_model_name]

labels_gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    random_state=SEED,
    n_init=10
).fit_predict(Z_best)

ari = adjusted_rand_score(
    labels_consensus, labels_gmm)
print(f'  Best latent : {best_model_name}')
print(f'  GMM ARI     : {ari:.4f}')

# Add to metadata
cd_meta['cluster']   = labels_consensus
cd_meta['stability'] = stability
cd_meta['subtype']   = [
    'Hyperinflammatory'
    if l == hyper_label
    else 'Quiescent'
    for l in labels_consensus]

# Save
np.save(
    f'{DATA_DIR}/'
    'ibdome_cd_consensus_v2.npy',
    C_mat)
np.save(
    f'{DATA_DIR}/'
    'ibdome_cd_labels_v2.npy',
    labels_consensus)
cd_meta.to_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_final_v2.csv',
    index=False)
print('Clustering results saved')

Consensus clustering...
  AE: (201, 16)
  DAE: (201, 16)
  VAE: (201, 64)
  BetaVAE: (201, 32)
  PCA: (201, 50)

Clustering solutions:
  AE_kmeans: sil=0.555 sizes=[142, 59]
  AE_ward: sil=0.555 sizes=[59, 142]
  AE_hdbscan: sil=0.560 sizes=[18, 127]
  DAE_kmeans: sil=0.566 sizes=[141, 60]
  DAE_ward: sil=0.564 sizes=[61, 140]
  DAE_hdbscan: sil=0.629 sizes=[29, 117]
  VAE_kmeans: sil=0.518 sizes=[153, 48]
  VAE_ward: sil=0.534 sizes=[177, 24]
  BetaVAE_kmeans: sil=0.399 sizes=[125, 76]
  BetaVAE_ward: sil=0.407 sizes=[145, 56]
  PCA_kmeans: sil=0.147 sizes=[94, 107]
  PCA_ward: sil=0.144 sizes=[137, 64]

Total solutions: 12
Building consensus matrix...

Consensus results:
  Silhouette : 0.7359
  Stability  : 0.8021
  Unstable   : 9
  Sizes      : [61, 140]
  C1 = Hyperinflammatory (n=61)
  C2 = Quiescent (n=140)

GMM validation...
  Best latent : DAE
  GMM ARI     : 0.8254
Clustering results saved


## Section 10 — Clinical Characterisation

Test whether identified subtypes differ
on clinical and demographic variables.

If subtypes differ significantly on age
or sex then the clustering may reflect
demographics rather than disease biology.

All variables tested:
- Age (continuous) — Mann-Whitney U
- Sex (categorical) — Chi-squared
- Hypertension (categorical) — Chi-squared
- Diabetes (categorical) — Chi-squared
- Smoking (categorical) — Chi-squared

Correction: Benjamini-Hochberg FDR

In [11]:
print('Clinical characterisation...')

clin_vars = {
    'age'                              : 'continuous',
    'sex'                              : 'categorical',
    'preexisting_arterial_hypertension': 'categorical',
    'preexisting_diabetes_mellitus'    : 'categorical',
    'riskfactor_nicotine'              : 'categorical',
}

clin_results = []
for var, vtype in clin_vars.items():
    col = next(
        (c for c in cd_meta.columns
         if var.lower() in c.lower()),
        None)
    if col is None:
        continue
    g0 = cd_meta.loc[
        labels_consensus == 0, col
    ].dropna()
    g1 = cd_meta.loc[
        labels_consensus == 1, col
    ].dropna()
    if len(g0) < 3 or len(g1) < 3:
        continue
    if vtype == 'continuous':
        _, p = mannwhitneyu(
            pd.to_numeric(
                g0, errors='coerce'
            ).dropna(),
            pd.to_numeric(
                g1, errors='coerce'
            ).dropna(),
            alternative='two-sided')
        test   = 'Mann-Whitney U'
        g0_num = pd.to_numeric(
            g0, errors='coerce')
        g1_num = pd.to_numeric(
            g1, errors='coerce')
        c0_val = (
            f'{g0_num.mean():.1f} '
            f'+- {g0_num.std():.1f}')
        c1_val = (
            f'{g1_num.mean():.1f} '
            f'+- {g1_num.std():.1f}')
    else:
        try:
            ct = pd.crosstab(
                labels_consensus,
                cd_meta[col])
            _, p, _, _ = \
                stats.chi2_contingency(ct)
        except Exception:
            p = 1.0
        test   = 'Chi-squared'
        c0_val = str(
            g0.value_counts().to_dict())
        c1_val = str(
            g1.value_counts().to_dict())
    sig = 'YES' if p < FDR_THRESH \
        else 'ns'
    clin_results.append({
        'Variable'   : col,
        'Test'       : test,
        'C1_value'   : c0_val,
        'C2_value'   : c1_val,
        'p_value'    : round(p, 5),
        'Significant': sig,
    })
    print(f'  {col:<40} '
           f'p={p:.4f} {sig}')

clin_df = pd.DataFrame(clin_results)
clin_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_clinical_v2.csv',
    index=False)
print('Clinical results saved')

Clinical characterisation...
  age                                      p=0.9979 ns
  sex                                      p=0.3234 ns
  preexisting_arterial_hypertension        p=0.7403 ns
  preexisting_diabetes_mellitus            p=0.3889 ns
  riskfactor_nicotine                      p=1.0000 ns
Clinical results saved


## Section 11 — Differential Abundance

Identify proteins significantly
different between hyperinflammatory
and quiescent subtypes.

**Method:** Welch t-test with
Benjamini-Hochberg FDR correction.

Welch t-test used instead of limma
because IBDome has only 61 proteins —
the empirical Bayes variance shrinkage
of limma is less critical at this scale.

**Threshold:** FDR < 0.05

logFC is defined as mean(C2) - mean(C1)
Positive logFC = higher in C2
Negative logFC = higher in C1

In [12]:
print('Differential abundance...')

de_results = []
g0_mask    = labels_consensus == 0
g1_mask    = labels_consensus == 1

for j, prot in enumerate(prot_cols):
    g0 = X_int[g0_mask, j]
    g1 = X_int[g1_mask, j]
    if len(g0) < 3 or len(g1) < 3:
        continue
    _, p = mannwhitneyu(
        g0, g1,
        alternative='two-sided')
    fc = g1.mean() - g0.mean()
    de_results.append({
        'protein' : prot,
        'logFC'   : round(fc, 5),
        'p_value' : p,
        'mean_C1' : round(g0.mean(), 5),
        'mean_C2' : round(g1.mean(), 5),
        'std_C1'  : round(g0.std(), 5),
        'std_C2'  : round(g1.std(), 5),
    })

de_df = pd.DataFrame(de_results)

_, fdr, _, _ = multipletests(
    de_df['p_value'],
    method='fdr_bh')
de_df['adj_p_value'] = fdr
de_df['significant'] = fdr < FDR_THRESH
de_df = de_df.sort_values(
    'adj_p_value')

n_sig   = de_df['significant'].sum()
n_up_c2 = (
    (de_df['significant']) &
    (de_df['logFC'] > 0)).sum()
n_up_c1 = (
    (de_df['significant']) &
    (de_df['logFC'] < 0)).sum()

print(f'  Total tested : {len(de_df)}')
print(f'  Significant  : {n_sig}')
print(f'  Higher in C2 : {n_up_c2}')
print(f'  Higher in C1 : {n_up_c1}')
print(f'\n  Top 10 higher in C2:')
for _, r in de_df[
        de_df['logFC'] > 0
].head(10).iterrows():
    print(f'    {r["protein"]:<15} '
           f'logFC={r["logFC"]:+.3f} '
           f'FDR={r["adj_p_value"]:.2e}')

de_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_de_v2.csv',
    index=False)
print('\nDE results saved')

Differential abundance...
  Total tested : 61
  Significant  : 0
  Higher in C2 : 0
  Higher in C1 : 0

  Top 10 higher in C2:
    IL6             logFC=+0.240 FDR=8.77e-01
    IL-17A          logFC=+0.215 FDR=8.77e-01
    CXCL5           logFC=+0.190 FDR=8.77e-01
    FGF-21          logFC=+0.249 FDR=8.77e-01
    IL18            logFC=+0.186 FDR=8.77e-01
    4E-BP1          logFC=+0.161 FDR=8.77e-01
    CD8A            logFC=+0.156 FDR=9.70e-01
    CXCL1           logFC=+0.033 FDR=9.70e-01
    OSM             logFC=+0.015 FDR=9.70e-01
    AXIN1           logFC=+0.048 FDR=9.70e-01

DE results saved


## Section 12 — Pathway Enrichment

Over-representation analysis (ORA)
performed against MSigDB Hallmark,
KEGG, and Reactome gene sets.

**Method:** Fisher exact test via
GSEApy Enrichr function.

**Background:** All 61 proteins
measured in IBDome panel.

**Test set:** Significantly upregulated
proteins in hyperinflammatory subtype
(FDR < 0.05, logFC > 0).

Note: ORA was used rather than GSEA
because we have a defined significant
protein list rather than a continuous
ranked list.

Reference: Liberzon et al. (2015),
Fang et al. (2023)

In [17]:
print('Pathway enrichment (ORA)...')

sig_up_prots = de_df[
    (de_df['significant']) &
    (de_df['logFC'] > 0)
]['protein'].tolist()

print(f'  Input proteins: '
       f'{len(sig_up_prots)}')

ora_results = {}
for gene_set in [
        'MSigDB_Hallmark_2020',
        'KEGG_2021_Human',
        'Reactome_2022']:
    try:
        enr = gp.enrichr(
            gene_list=sig_up_prots,
            gene_sets=[gene_set],
            background=prot_cols,
            outdir=None,
            verbose=False)
        if enr.results is None:
            continue
        res = enr.results.copy()
        res = res[
            res['Adjusted P-value']
            < FDR_THRESH
        ].sort_values('Adjusted P-value')
        ora_results[gene_set] = res
        label = gene_set.split('_')[0]
        res.to_csv(
            f'{TABLES_DIR}/'
            f'ibdome_cd_ora_{label}'
            f'_v2.csv',
            index=False)
        print(f'  {gene_set}: '
               f'{len(res)} significant')
        for _, r in res.head(3).iterrows():
            fdr_val = r['Adjusted P-value']
            term    = r['Term'][:45]
            print(f'    {term} '
                   f'FDR={fdr_val:.2e}')
    except Exception as e:
        print(f'  {gene_set}: {e}')

Pathway enrichment (ORA)...
  Input proteins: 0
  MSigDB_Hallmark_2020: Gene list cannot be empty
  KEGG_2021_Human: Gene list cannot be empty
  Reactome_2022: Gene list cannot be empty


## Section 13 — Random Forest Triangulation

Random Forest classifier trained to
predict subtype labels from protein
profiles.

**Purpose:** Independent validation of
subtype separability using a machine
learning approach.

**Setup:**
- 500 decision trees
- Square-root feature selection
- 5-fold stratified cross-validation
- Balanced class weights

**Triangulation:** Top RF proteins
compared with limma significant proteins
to assess methodological agreement.
High overlap = robust signal.

Reference: Breiman (2001)

In [18]:
print('Random Forest triangulation...')

y_rf = (labels_consensus ==
        hyper_label).astype(int)

rf_model = RandomForestClassifier(
    n_estimators=RF_N_TREES,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
    class_weight='balanced')

cv   = StratifiedKFold(
    n_splits=RF_CV_FOLDS,
    shuffle=True,
    random_state=SEED)
aucs = []

for fold, (tr, te) in enumerate(
        cv.split(X_int, y_rf)):
    rf_model.fit(X_int[tr], y_rf[tr])
    prob = rf_model.predict_proba(
        X_int[te])[:, 1]
    auc  = roc_auc_score(
        y_rf[te], prob)
    aucs.append(auc)
    print(f'  Fold {fold+1}: '
           f'AUC={auc:.4f}')

mean_auc = np.mean(aucs)
std_auc  = np.std(aucs)
print(f'\n  CV AUC = {mean_auc:.4f} '
       f'+- {std_auc:.4f}')

# Final fit for importance
rf_model.fit(X_int, y_rf)
imp   = rf_model.feature_importances_
rf_df = pd.DataFrame({
    'protein'  : prot_cols,
    'gini_imp' : imp,
}).sort_values(
    'gini_imp',
    ascending=False
).reset_index(drop=True)

# Triangulation
de_sig_set = set(
    de_df[de_df['significant']][
        'protein'].tolist())
top20_set  = set(
    rf_df.head(20)['protein'].tolist())
top30_set  = set(
    rf_df.head(30)['protein'].tolist())
overlap20  = top20_set & de_sig_set
overlap30  = top30_set & de_sig_set

print(f'  RF-DE top20: '
       f'{len(overlap20)}/20 '
       f'({len(overlap20)/20*100:.0f}%)')
print(f'  RF-DE top30: '
       f'{len(overlap30)}/30 '
       f'({len(overlap30)/30*100:.0f}%)')

rf_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_rf_v2.csv',
    index=False)
print('RF results saved')

Random Forest triangulation...
  Fold 1: AUC=0.4890
  Fold 2: AUC=0.4821
  Fold 3: AUC=0.5089
  Fold 4: AUC=0.3720
  Fold 5: AUC=0.5565

  CV AUC = 0.4817 +- 0.0607
  RF-DE top20: 0/20 (0%)
  RF-DE top30: 0/30 (0%)
RF results saved


## Section 14 — PPI Network Analysis

Protein-protein interaction networks
constructed from top differentially
abundant proteins using the STRING
database.

**Threshold:** Confidence score >= 0.40
(medium to high confidence interactions)

**Metrics computed:**
- Degree centrality: number of direct
  connections per protein
- Betweenness centrality: fraction of
  shortest paths passing through node

Hub proteins identified as those with
highest degree and betweenness scores.
These represent therapeutic target
candidates.

Reference: Szklarczyk et al. (2023)

In [19]:
print('PPI network analysis...')

top_prots = rf_df.head(
    TOP_N_PPI)['protein'].tolist()

G      = None
ppi_df = None

try:
    import requests
    prot_str = '%0d'.join(top_prots)
    url = (
        'https://string-db.org/api/'
        'json/network'
        f'?identifiers={prot_str}'
        '&species=9606'
        f'&required_score='
        f'{int(STRING_SCORE * 1000)}'
        '&caller_identity=IBD_MSc_UoB')
    resp = requests.get(
        url, timeout=30)
    if resp.status_code == 200:
        ppi_df = pd.DataFrame(
            resp.json())
        ppi_df.to_csv(
            f'{TABLES_DIR}/'
            'ibdome_cd_ppi_v2.csv',
            index=False)
        G = nx.Graph()
        for _, row in ppi_df.iterrows():
            a = row.get(
                'preferredName_A', '')
            b = row.get(
                'preferredName_B', '')
            w = float(row.get(
                'score', STRING_SCORE))
            if a and b:
                G.add_edge(
                    a, b, weight=w)
        deg = dict(G.degree())
        bc  = nx.betweenness_centrality(
            G)
        print(f'  Nodes : '
               f'{G.number_of_nodes()}')
        print(f'  Edges : '
               f'{G.number_of_edges()}')
        print('  Top 5 hubs:')
        for prot, d in sorted(
                deg.items(),
                key=lambda x: x[1],
                reverse=True)[:5]:
            print(f'    {prot}: '
                   f'degree={d} '
                   f'BC={bc.get(prot,0):.3f}')
    else:
        print(f'  STRING API: '
               f'{resp.status_code}')
except Exception as e:
    print(f'  PPI error: {e}')

# Fallback to saved file
if ppi_df is None:
    saved = (f'{TABLES_DIR}/'
              'ibdome_cd_ppi_C2_top30.csv')
    if os.path.exists(saved):
        ppi_df = pd.read_csv(
            saved, encoding='latin1')
        G = nx.Graph()
        for _, row in ppi_df.iterrows():
            a = row.get(
                'preferredName_A', '')
            b = row.get(
                'preferredName_B', '')
            if a and b:
                G.add_edge(
                    a, b,
                    weight=float(
                        row.get(
                            'score',
                            STRING_SCORE)))
        print(f'  Loaded saved PPI: '
               f'{G.number_of_nodes()} '
               f'nodes')

PPI network analysis...
  Nodes : 28
  Edges : 165
  Top 5 hubs:
    IL6: degree=25 BC=0.178
    IL17A: degree=20 BC=0.040
    CD8A: degree=19 BC=0.070
    CCL20: degree=19 BC=0.042
    CCL11: degree=19 BC=0.049


## Section 15 — Chemokine x Vascular Validation

Pathway activity scores computed for
three predefined protein groups and
correlated to assess co-activation
patterns.

**Score definition:**
Mean INT-normalised expression of
available proteins from each group
present in the IBDome panel.

**UK Biobank reference:**
Chemokine x Vascular r = 0.749

**Validation criterion:**
IBDome r should be close to UKB r.
Delta < 0.05 indicates successful
replication of co-activation pattern.

In [20]:
print('Chemokine x Vascular validation...')

def pathway_score(X, prot_list,
                   available):
    present = [
        p for p in prot_list
        if p in available]
    if not present:
        return np.zeros(
            X.shape[0]), []
    idx = [available.index(p)
            for p in present]
    return X[:, idx].mean(axis=1), \
        present

sc_chemo, used_chemo = pathway_score(
    X_int, CHEMOKINE_LIST, prot_cols)
sc_vasc,  used_vasc  = pathway_score(
    X_int, VASCULAR_LIST, prot_cols)
sc_cyto,  used_cyto  = pathway_score(
    X_int, CYTOKINE_LIST, prot_cols)

r_cv, p_cv = stats.pearsonr(
    sc_chemo, sc_vasc)
r_cc, p_cc = stats.pearsonr(
    sc_chemo, sc_cyto)

print(f'  Chemokine proteins : '
       f'{len(used_chemo)} '
       f'-> {used_chemo}')
print(f'  Vascular proteins  : '
       f'{len(used_vasc)} '
       f'-> {used_vasc}')
print(f'  Cytokine proteins  : '
       f'{len(used_cyto)} '
       f'-> {used_cyto}')
print(f'\n  Chemo x Vasc  : '
       f'r={r_cv:.4f} p={p_cv:.4e}')
print(f'  Chemo x Cyto  : '
       f'r={r_cc:.4f} p={p_cc:.4e}')
print(f'  UKB reference : r=0.749')
print(f'  Delta         : '
       f'{abs(r_cv - 0.749):.4f}')

val_dict = {
    'cohort'                   : COHORT,
    'disease'                  : DISEASE,
    'chemokine_vascular_r'     : round(r_cv, 5),
    'chemokine_vascular_p'     : round(p_cv, 10),
    'chemokine_cytokine_r'     : round(r_cc, 5),
    'chemokine_cytokine_p'     : round(p_cc, 10),
    'ukb_reference_r'          : 0.749,
    'delta_r'                  : round(
        abs(r_cv - 0.749), 5),
    'n_chemokine_used'         : len(used_chemo),
    'n_vascular_used'          : len(used_vasc),
    'n_cytokine_used'          : len(used_cyto),
    'chemokine_proteins_used'  : used_chemo,
    'vascular_proteins_used'   : used_vasc,
    'cytokine_proteins_used'   : used_cyto,
}
with open(
        f'{DATA_DIR}/'
        'ibdome_cd_validation_v2.json',
        'w') as f:
    json.dump(val_dict, f, indent=2)
print('Validation metrics saved')

Chemokine x Vascular validation...
  Chemokine proteins : 15 -> ['IL8', 'MCP-3', 'MCP-1', 'CXCL11', 'CXCL9', 'CXCL1', 'CCL4', 'CCL19', 'CXCL5', 'CCL3', 'CXCL6', 'CXCL10', 'CCL28', 'CCL25', 'CCL20']
  Vascular proteins  : 6 -> ['VEGFA', 'HGF', 'FGF-21', 'FGF-19', 'FGF-5', 'FGF-23']
  Cytokine proteins  : 8 -> ['IL6', 'IL-17C', 'IL-17A', 'OSM', 'IL18', 'IL10', 'TNF', 'IFN-gamma']

  Chemo x Vasc  : r=0.7422 p=1.9288e-36
  Chemo x Cyto  : r=0.6814 p=8.7050e-29
  UKB reference : r=0.749
  Delta         : 0.0068
Validation metrics saved


## Section 16 — Figures

Three publication-quality figures:

**Figure 1 — Clustering overview:**
UMAP, PCA, stability, consensus matrix,
silhouette distribution, cluster sizes.

**Figure 2 — Biological characterisation:**
Volcano plot and ORA dot plot.

**Figure 3 — Validation:**
Chemokine x Vascular scatter and
cross-cohort metric comparison.

In [21]:
print('Generating figures...')

# Colour palette
PALETTE = {}
NAMES   = {}
for c in range(N_CLUSTERS):
    if c == hyper_label:
        PALETTE[c] = C2_COL
        NAMES[c]   = 'Hyperinflammatory'
    else:
        PALETTE[c] = C1_COL
        NAMES[c]   = 'Quiescent'

def clean_term(t):
    t = re.sub(r'HALLMARK_', '', str(t))
    t = re.sub(
        r'\s+R-HSA-\d+', '', t)
    t = t.replace(
        '_', ' ').strip().title()
    fixes = {
        'Tnf Alpha Signaling Via Nfkb':
            'TNF/NF-kB',
        'Il6 Jak Stat3 Signaling':
            'IL-6/JAK/STAT3',
        'Interferon Gamma Response':
            'IFN-gamma Response',
        'Inflammatory Response':
            'Inflammatory Response',
        'Allograft Rejection':
            'Allograft Rejection',
        'Epithelial Mesenchymal '
        'Transition': 'EMT',
    }
    for k, v in fixes.items():
        if k.lower() in t.lower():
            return v
    return t[:42]

# ── Figure 1: Clustering ──────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('white')
gs  = gridspec.GridSpec(
    2, 3, figure=fig,
    hspace=0.40, wspace=0.35)

# Panel A: UMAP
ax = fig.add_subplot(gs[0, 0])
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
try:
    red = umap.UMAP(
        n_neighbors=15,
        min_dist=0.10,
        n_components=2,
        random_state=SEED)
    emb = red.fit_transform(Z_best)
    for c in range(N_CLUSTERS):
        m = labels_consensus == c
        ax.scatter(
            emb[m, 0], emb[m, 1],
            c=PALETTE[c], s=55,
            alpha=0.85,
            linewidths=0.4,
            edgecolors='white',
            label=(f'{NAMES[c]} '
                    f'(n={m.sum()})'),
            zorder=3)
    ax.legend(fontsize=10,
               framealpha=0.95)
except Exception as e:
    ax.text(0.5, 0.5,
             f'UMAP error:\n{e}',
             ha='center',
             va='center',
             transform=ax.transAxes,
             fontsize=9)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    f'(A) IBDome CD  k={N_CLUSTERS}\n'
    f'sil={sil_final:.3f}  '
    f'stab={stab_mean:.3f}',
    fontsize=11, fontweight='700',
    loc='left')

# Panel B: PCA
ax  = fig.add_subplot(gs[0, 1])
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
pca2 = PCA(n_components=2,
            random_state=SEED)
Xp   = pca2.fit_transform(X_int)
var2 = (
    pca2.explained_variance_ratio_
    * 100)
for c in range(N_CLUSTERS):
    m = labels_consensus == c
    ax.scatter(
        Xp[m, 0], Xp[m, 1],
        c=PALETTE[c], s=35,
        alpha=0.70, linewidths=0,
        label=NAMES[c], zorder=3)
ax.set_xlabel(
    f'PC1 ({var2[0]:.1f}%)',
    fontsize=11)
ax.set_ylabel(
    f'PC2 ({var2[1]:.1f}%)',
    fontsize=11)
ax.set_title(
    '(B) PCA',
    fontsize=11, fontweight='700',
    loc='left')

# Panel C: Stability
ax = fig.add_subplot(gs[0, 2])
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
if 'emb' in dir():
    sc = ax.scatter(
        emb[:, 0], emb[:, 1],
        c=stability,
        cmap='RdYlGn',
        vmin=0.5, vmax=1.0,
        s=55, alpha=0.85,
        linewidths=0, zorder=3)
    plt.colorbar(
        sc, ax=ax,
        label='Stability',
        shrink=0.8)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    f'(C) Stability\n'
    f'Mean={stab_mean:.3f}  '
    f'Unstable={n_unstable}',
    fontsize=11, fontweight='700',
    loc='left')

# Panel D: Consensus matrix
ax  = fig.add_subplot(gs[1, 0])
idx = np.argsort(labels_consensus)
C_s = C_mat[idx][:, idx]
l_s = labels_consensus[idx]
im  = ax.imshow(
    C_s, cmap='Blues',
    vmin=0, vmax=1,
    aspect='auto',
    interpolation='nearest')
bnd = (l_s == 0).sum() - 0.5
ax.axhline(bnd, color='white',
            lw=2.5)
ax.axvline(bnd, color='white',
            lw=2.5)
plt.colorbar(im, ax=ax,
              label='Co-occurrence',
              shrink=0.7)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    f'(D) Consensus matrix\n'
    f'{len(all_solutions)} solutions',
    fontsize=11, fontweight='700',
    loc='left')

# Panel E: Silhouette
ax      = fig.add_subplot(gs[1, 1])
sil_per = silhouette_samples(
    D_mat, labels_consensus,
    metric='precomputed')
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
y_lower = 10
for c in range(N_CLUSTERS):
    mask    = labels_consensus == c
    c_sils  = np.sort(sil_per[mask])
    sz      = mask.sum()
    y_upper = y_lower + sz
    ax.barh(
        range(y_lower, y_upper),
        c_sils, height=1.0,
        color=PALETTE[c],
        alpha=0.85,
        edgecolor='none')
    ax.text(
        -0.02,
        (y_lower + y_upper) / 2,
        f'{NAMES[c]}\n(n={sz})',
        ha='right', va='center',
        fontsize=8.5,
        fontweight='700',
        color=PALETTE[c])
    y_lower = y_upper + 10
ax.axvline(
    sil_per.mean(),
    color='black', lw=1.5,
    linestyle='--',
    label=f'Mean='
          f'{sil_per.mean():.3f}')
ax.axvline(
    0, color='#888',
    lw=0.8, alpha=0.4)
ax.set_xlabel(
    'Silhouette', fontsize=11)
ax.set_yticks([])
ax.legend(fontsize=9)
ax.set_title(
    '(E) Per-patient silhouette',
    fontsize=11, fontweight='700',
    loc='left')

# Panel F: Sizes
ax = fig.add_subplot(gs[1, 2])
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.bar(
    [NAMES[c]
     for c in range(N_CLUSTERS)],
    [cluster_sizes[c]
     for c in range(N_CLUSTERS)],
    color=[PALETTE[c]
            for c in range(N_CLUSTERS)],
    alpha=0.88,
    edgecolor='white')
for c in range(N_CLUSTERS):
    n = cluster_sizes[c]
    ax.text(
        c, n + 1,
        f'n={n}\n({n/N*100:.1f}%)',
        ha='center', va='bottom',
        fontsize=10,
        fontweight='700')
ax.set_ylabel('Patients', fontsize=11)
ax.set_title(
    '(F) Cluster sizes',
    fontsize=11, fontweight='700',
    loc='left')

plt.suptitle(
    f'IBDome CD -- Proteomic subtypes\n'
    f'n={N}  k={N_CLUSTERS}  '
    f'sil={sil_final:.3f}  '
    f'stab={stab_mean:.3f}  '
    f'GMM ARI={ari:.3f}',
    fontsize=13, fontweight='900',
    y=1.01)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/'
    'ibdome_cd_clustering_v2.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('  Clustering figure saved')

# ── Figure 2: Volcano + ORA ───────────
fig, axes = plt.subplots(
    1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

ax = axes[0]
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
de_df['log_q'] = -np.log10(
    de_df['adj_p_value'].clip(1e-50))
sig_up  = (
    de_df['significant'] &
    (de_df['logFC'] > 0))
sig_dn  = (
    de_df['significant'] &
    (de_df['logFC'] < 0))
ns_mask = ~de_df['significant']
ax.scatter(
    de_df.loc[ns_mask, 'logFC'],
    de_df.loc[ns_mask, 'log_q'],
    c='#CCCCCC', s=15,
    alpha=0.35, linewidths=0,
    zorder=1)
ax.scatter(
    de_df.loc[sig_up, 'logFC'],
    de_df.loc[sig_up, 'log_q'],
    c=C2_COL, s=30, alpha=0.85,
    linewidths=0.3,
    edgecolors='white', zorder=3)
ax.scatter(
    de_df.loc[sig_dn, 'logFC'],
    de_df.loc[sig_dn, 'log_q'],
    c=C1_COL, s=30, alpha=0.85,
    linewidths=0.3,
    edgecolors='white', zorder=3)
for _, row in de_df[
        sig_up].head(8).iterrows():
    ax.annotate(
        row['protein'],
        (row['logFC'], row['log_q']),
        fontsize=8, style='italic',
        fontweight='600',
        color=C2_COL,
        xytext=(6, 3),
        textcoords='offset points',
        arrowprops=dict(
            arrowstyle='-',
            color=C2_COL,
            alpha=0.4, lw=0.7))
for _, row in de_df[
        sig_dn].head(6).iterrows():
    ax.annotate(
        row['protein'],
        (row['logFC'], row['log_q']),
        fontsize=8, style='italic',
        fontweight='600',
        color=C1_COL,
        xytext=(-32, 3),
        textcoords='offset points',
        arrowprops=dict(
            arrowstyle='-',
            color=C1_COL,
            alpha=0.4, lw=0.7))
ax.axhline(
    -np.log10(FDR_THRESH),
    color='#555', linestyle='--',
    linewidth=1.2, alpha=0.7)
ax.axvline(
    0, color='#aaa',
    linewidth=0.8, alpha=0.5)
ax.text(
    0.97, 0.97,
    (f'Higher in C2: {sig_up.sum()}\n'
     f'Higher in C1: {sig_dn.sum()}\n'
     f'FDR < {FDR_THRESH} | Welch+BH'),
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=9.5,
    fontfamily='monospace',
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='white',
        edgecolor='#ccc',
        alpha=0.95))
ax.legend(handles=[
    Line2D([0], [0], marker='o',
            color='w',
            markerfacecolor=C2_COL,
            markersize=10,
            label=f'Higher in C2 '
                  f'(n={sig_up.sum()})'),
    Line2D([0], [0], marker='o',
            color='w',
            markerfacecolor=C1_COL,
            markersize=10,
            label=f'Higher in C1 '
                  f'(n={sig_dn.sum()})'),
    Line2D([0], [0], marker='o',
            color='w',
            markerfacecolor='#CCCCCC',
            markersize=10,
            label='Not significant')],
    fontsize=9.5,
    loc='upper left',
    framealpha=0.95)
ax.set_xlabel(
    'log2 Fold Change (C2 vs C1)',
    fontsize=12)
ax.set_ylabel(
    '-log10(FDR q-value)',
    fontsize=12)
ax.set_title(
    '(A) Differential Protein '
    'Abundance\nWelch t-test + BH  '
    f'FDR < {FDR_THRESH}',
    fontsize=12, fontweight='700',
    loc='left')

# ORA dot plot
ax = axes[1]
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='x', alpha=0.2,
        linestyle='--', color='#ccc')
ax.tick_params(length=0)

hallmark_key = 'MSigDB_Hallmark_2020'
if (hallmark_key in ora_results and
        len(ora_results[hallmark_key])
        > 0):
    hm = ora_results[
        hallmark_key].head(10).copy()
    hm['log_fdr'] = -np.log10(
        hm['Adjusted P-value'].clip(
            lower=1e-30))
    hm['Term_clean'] = hm[
        'Term'].apply(clean_term)
    hm = hm.sort_values(
        'log_fdr', ascending=True)

    def parse_overlap(ov):
        try:
            if '/' in str(ov):
                return int(
                    str(ov).split('/')[0])
            return int(float(str(ov)))
        except Exception:
            return 8

    ov_col = next(
        (c for c in
         ['Overlap', 'overlap',
          'Gene_count']
         if c in hm.columns), None)
    hm['n_genes'] = (
        hm[ov_col].apply(parse_overlap)
        if ov_col else 10)
    min_g  = hm['n_genes'].min()
    max_g  = hm['n_genes'].max()
    rng    = max(max_g - min_g, 1)
    sizes  = [
        100 + ((n - min_g) / rng) * 300
        for n in hm['n_genes']]
    sc2 = ax.scatter(
        hm['log_fdr'],
        range(len(hm)),
        s=sizes,
        c=hm['log_fdr'],
        cmap='YlOrRd',
        vmin=hm['log_fdr'].min(),
        vmax=hm['log_fdr'].max(),
        alpha=0.92,
        linewidths=0.8,
        edgecolors='white',
        zorder=4)
    for xi, (_, row) in enumerate(
            hm.iterrows()):
        ax.text(
            row['log_fdr'] + 0.10,
            xi,
            f'{row["log_fdr"]:.1f}',
            va='center', fontsize=9,
            color='#444',
            fontweight='600')
    ax.set_yticks(range(len(hm)))
    ax.set_yticklabels(
        hm['Term_clean'], fontsize=10.5)
    plt.colorbar(
        sc2, ax=ax,
        label='-log10(FDR)',
        shrink=0.6, pad=0.02)
    ax.axvline(
        -np.log10(FDR_THRESH),
        color='#888', linewidth=1.2,
        linestyle='--', alpha=0.7)
else:
    ax.text(
        0.5, 0.5,
        'No significant pathways',
        ha='center', va='center',
        transform=ax.transAxes,
        fontsize=12, color='#888')

ax.set_xlabel(
    '-log10(FDR q-value)',
    fontsize=12)
ax.set_title(
    '(B) Hallmark Pathway Enrichment\n'
    'Hyperinflammatory subtype',
    fontsize=12, fontweight='700',
    loc='left')

plt.suptitle(
    f'IBDome CD -- Biological '
    f'characterisation\n'
    f'n={N}  {n_sig} significant '
    f'proteins  RF AUC={mean_auc:.3f}',
    fontsize=13, fontweight='900',
    y=1.01)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/'
    'ibdome_cd_biology_v2.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('  Biology figure saved')

# ── Figure 3: Validation ──────────────
fig, axes = plt.subplots(
    1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

ax = axes[0]
ax.set_facecolor(BG_COL)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.12, linestyle='--',
        color='#ccc', zorder=0)
for c in range(N_CLUSTERS):
    m = labels_consensus == c
    ax.scatter(
        sc_chemo[m], sc_vasc[m],
        c=PALETTE[c], s=45,
        alpha=0.78,
        linewidths=0.4,
        edgecolors='white',
        label=(f'{NAMES[c]} '
                f'(n={m.sum()})'),
        zorder=3)
m_, b_ = np.polyfit(
    sc_chemo, sc_vasc, 1)
xr = np.linspace(
    sc_chemo.min(),
    sc_chemo.max(), 100)
ax.plot(
    xr, m_ * xr + b_,
    color='black', linewidth=2,
    linestyle='--', zorder=4)
ax.text(
    0.05, 0.95,
    (f'IBDome CD\n'
     f'r = {r_cv:.3f}\n'
     f'p < 0.0001\n\n'
     f'UKB reference\n'
     f'r = 0.749\n'
     f'Delta = {abs(r_cv-0.749):.3f}'),
    transform=ax.transAxes,
    va='top', fontsize=11,
    fontweight='700',
    bbox=dict(
        boxstyle='round,pad=0.5',
        facecolor='white',
        edgecolor='#DDD',
        alpha=0.95))
ax.set_xlabel(
    'Chemokine score', fontsize=12)
ax.set_ylabel(
    'Vascular score', fontsize=12)
ax.set_title(
    '(A) Chemokine x Vascular\n'
    'co-activation replicated',
    fontsize=12, fontweight='700',
    loc='left')
ax.legend(fontsize=10,
           framealpha=0.95)

ax = axes[1]
ax.set_facecolor('white')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.grid(axis='y', alpha=0.2,
        linestyle='--', color='#ccc')
ax.tick_params(length=0)
metrics  = ['Silhouette',
             'Stability', 'RF AUC']
ukb_vals = [0.846, 0.893, 0.983]
ib_vals  = [sil_final, stab_mean,
             mean_auc]
x_pos    = np.arange(len(metrics))
w        = 0.30
ax.bar(x_pos - w/2, ukb_vals,
        width=w, color='#1B3A6B',
        alpha=0.88, label='UKB CD',
        edgecolor='white', zorder=3)
ax.bar(x_pos + w/2, ib_vals,
        width=w, color='#C96A1F',
        alpha=0.88, label='IBDome CD',
        edgecolor='white', zorder=3)
for xi, (u, b) in enumerate(
        zip(ukb_vals, ib_vals)):
    ax.text(xi - w/2, u + 0.005,
             f'{u:.3f}',
             ha='center', va='bottom',
             fontsize=10,
             color='#1B3A6B',
             fontweight='700')
    ax.text(xi + w/2, b + 0.005,
             f'{b:.3f}',
             ha='center', va='bottom',
             fontsize=10,
             color='#C96A1F',
             fontweight='700')
ax.set_xticks(x_pos)
ax.set_xticklabels(
    metrics, fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Value', fontsize=12)
ax.legend(fontsize=11,
           framealpha=0.95)
ax.set_title(
    '(B) UKB vs IBDome metrics',
    fontsize=12, fontweight='700',
    loc='left')

plt.suptitle(
    f'IBDome CD -- External Validation\n'
    f'Chemokine x Vascular r={r_cv:.3f}  '
    f'UKB ref=0.749  '
    f'Delta={abs(r_cv-0.749):.3f}',
    fontsize=13, fontweight='900',
    y=1.01)
plt.tight_layout()
plt.savefig(
    f'{FIGURES_DIR}/'
    'ibdome_cd_validation_v2.png',
    dpi=200, bbox_inches='tight',
    facecolor='white')
plt.close()
print('  Validation figure saved')

Generating figures...
  Clustering figure saved
  Biology figure saved
  Validation figure saved


## Section 17 — Summary Tables

Save all key results in a single
summary CSV for easy reference
and thesis reporting.

In [22]:
print('Saving summary...')

summary = {
    'Cohort'              : COHORT,
    'Disease'             : "Crohn's Disease",
    'Country'             : 'Germany',
    'N_patients'          : N,
    'N_proteins'          : P,
    'k'                   : N_CLUSTERS,
    'Silhouette'          : round(sil_final, 4),
    'Stability'           : round(stab_mean, 4),
    'N_unstable'          : int(n_unstable),
    'GMM_ARI'             : round(ari, 4),
    'N_hyperinflam'       : int(
        cluster_sizes[hyper_label]),
    'N_quiescent'         : int(
        cluster_sizes[quiet_label]),
    'DE_significant'      : int(n_sig),
    'DE_up_hyperinflam'   : int(n_up_c2),
    'DE_up_quiescent'     : int(n_up_c1),
    'RF_AUC_mean'         : round(mean_auc, 4),
    'RF_AUC_std'          : round(std_auc, 4),
    'RF_DE_overlap_top20' : len(overlap20),
    'RF_DE_overlap_top30' : len(overlap30),
    'Chemo_Vasc_r'        : round(r_cv, 4),
    'UKB_reference_r'     : 0.749,
    'Delta_r'             : round(
        abs(r_cv - 0.749), 4),
    'N_chemo_prots_used'  : len(used_chemo),
    'N_vasc_prots_used'   : len(used_vasc),
}

pd.DataFrame([summary]).to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_summary_v2.csv',
    index=False)

print('\nSUMMARY')
print('=' * 55)
for k, v in summary.items():
    print(f'  {k:<30} {v}')

print('\n' + '=' * 55)
print('IBDome CD Pipeline COMPLETE')
print('=' * 55)
print(f'\nFigures : {FIGURES_DIR}')
print(f'Tables  : {TABLES_DIR}')
print(f'Data    : {DATA_DIR}')

output_files = [
    (DATA_DIR,    'ibdome_cd_preprocessed_v2.csv'),
    (DATA_DIR,    'ibdome_cd_meta_v2.csv'),
    (DATA_DIR,    'ibdome_cd_final_v2.csv'),
    (DATA_DIR,    'ibdome_cd_consensus_v2.npy'),
    (DATA_DIR,    'ibdome_cd_labels_v2.npy'),
    (DATA_DIR,    'ibdome_cd_validation_v2.json'),
    (TABLES_DIR,  'ibdome_cd_clinical_v2.csv'),
    (TABLES_DIR,  'ibdome_cd_de_v2.csv'),
    (TABLES_DIR,  'ibdome_cd_rf_v2.csv'),
    (TABLES_DIR,  'ibdome_cd_summary_v2.csv'),
    (FIGURES_DIR, 'ibdome_cd_clustering_v2.png'),
    (FIGURES_DIR, 'ibdome_cd_biology_v2.png'),
    (FIGURES_DIR, 'ibdome_cd_validation_v2.png'),
]

print('\nOutput files:')
for directory, fname in output_files:
    path = f'{directory}/{fname}'
    if os.path.exists(path):
        sz = os.path.getsize(path) // 1024
        print(f'  SAVED    {fname} ({sz} KB)')
    else:
        print(f'  MISSING  {fname}')

Saving summary...

SUMMARY
  Cohort                         IBDome_CD
  Disease                        Crohn's Disease
  Country                        Germany
  N_patients                     201
  N_proteins                     61
  k                              2
  Silhouette                     0.7359
  Stability                      0.8021
  N_unstable                     9
  GMM_ARI                        0.8254
  N_hyperinflam                  61
  N_quiescent                    140
  DE_significant                 0
  DE_up_hyperinflam              0
  DE_up_quiescent                0
  RF_AUC_mean                    0.4817
  RF_AUC_std                     0.0607
  RF_DE_overlap_top20            0
  RF_DE_overlap_top30            0
  Chemo_Vasc_r                   0.7422
  UKB_reference_r                0.749
  Delta_r                        0.0068
  N_chemo_prots_used             15
  N_vasc_prots_used              6

IBDome CD Pipeline COMPLETE

Figures : /rds/homes/j/jxt554

In [33]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import \
    RandomForestClassifier
from sklearn.model_selection import \
    StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    silhouette_score,
    adjusted_rand_score)
from sklearn.mixture import \
    GaussianMixture
from statsmodels.stats.multitest import \
    multipletests
import os
import json

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'
SEED       = 42
np.random.seed(SEED)

print('IBDome CD — Original Setup')
print('='*55)

# ── Load EXACT original files ─────────
X_int = pd.read_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_preprocessed.csv')
prot_cols = X_int.columns.tolist()
X_int     = X_int.values

meta = pd.read_csv(
    f'{DATA_DIR}/ibdome_cd_final.csv')

# Use DEC consensus labels
labels_consensus = np.load(
    f'{DATA_DIR}/'
    'ibdome_labels_dec_consensus.npy')
C_mat = np.load(
    f'{DATA_DIR}/'
    'ibdome_consensus_dec.npy')
stab_orig = np.load(
    f'{DATA_DIR}/'
    'ibdome_stability_dec.npy')

N = len(X_int)
P = len(prot_cols)

print(f'X shape     : {X_int.shape}')
print(f'Labels      : '
       f'{np.bincount(labels_consensus)}')
print(f'N match     : '
       f'{N == len(labels_consensus)}')

# ── Metrics ───────────────────────────
D_mat = np.clip(1-C_mat, 0, None)
np.fill_diagonal(D_mat, 0)
sil_final  = silhouette_score(
    D_mat, labels_consensus,
    metric='precomputed')
stab_mean  = stab_orig.mean()
n_unstable = (stab_orig < 0.5).sum()
cluster_sizes = np.bincount(
    labels_consensus)

print(f'\nClustering:')
print(f'  Silhouette : {sil_final:.4f}')
print(f'  Stability  : {stab_mean:.4f}')
print(f'  Unstable   : {n_unstable}')
print(f'  Sizes      : '
       f'{cluster_sizes.tolist()}')

# ── Identify hyperinflam ──────────────
# From original code:
# C1=0=quiescent C2=1=hyperinflammatory
# Confirmed by chemokine scores
# and DE results showing up in C2

inflam_prots = [
    'TNF','IL6','IL8','IL10',
    'HGF','OSM','CXCL10',
    'CXCL9','IFN-gamma','CCL4',
    'MCP-1','MCP-3','VEGFA']
found_idx = [
    prot_cols.index(p)
    for p in inflam_prots
    if p in prot_cols]

sc0 = X_int[
    labels_consensus==0][:,
    found_idx].mean()
sc1 = X_int[
    labels_consensus==1][:,
    found_idx].mean()

print(f'\nInflammatory scores:')
print(f'  C0 (n={cluster_sizes[0]}): '
       f'{sc0:.4f}')
print(f'  C1 (n={cluster_sizes[1]}): '
       f'{sc1:.4f}')

hyper_label = 0 if sc0>sc1 else 1
quiet_label = 1 - hyper_label
print(f'  Hyperinflam: '
       f'C{hyper_label+1} '
       f'(n={cluster_sizes[hyper_label]})')

# Check with HGF
if 'HGF' in prot_cols:
    hi = prot_cols.index('HGF')
    h  = [X_int[labels_consensus==c,
                  hi].mean()
           for c in range(2)]
    print(f'  HGF C0={h[0]:.4f} '
           f'C1={h[1]:.4f}')
    print(f'  HGF higher in '
           f'C{1 if h[0]>h[1] else 2}')

# ── DE with Welch t-test ──────────────
print('\nDifferential abundance...')
de_results = []
X1 = X_int[labels_consensus==0]
X2 = X_int[labels_consensus==1]

for j, prot in enumerate(prot_cols):
    t, p = stats.ttest_ind(
        X1[:,j], X2[:,j],
        equal_var=False)
    fc = X2[:,j].mean() - X1[:,j].mean()
    de_results.append({
        'protein'   : prot,
        'logFC'     : round(fc, 5),
        't_stat'    : round(t, 5),
        'p_value'   : p,
        'mean_C1'   : round(
            X1[:,j].mean(), 5),
        'mean_C2'   : round(
            X2[:,j].mean(), 5),
    })

de_df = pd.DataFrame(de_results)
_, fdr, _, _ = multipletests(
    de_df['p_value'],
    method='fdr_bh')
de_df['adj_p_value'] = fdr
de_df['significant'] = fdr < 0.05
de_df = de_df.sort_values('adj_p_value')

n_sig   = de_df['significant'].sum()
n_up_c2 = ((de_df['significant']) &
            (de_df['logFC']>0)).sum()
n_up_c1 = ((de_df['significant']) &
            (de_df['logFC']<0)).sum()

print(f'  Significant : {n_sig}')
print(f'  Higher C2   : {n_up_c2}')
print(f'  Higher C1   : {n_up_c1}')
print(f'\n  Top 10:')
for _, r in de_df.head(10).iterrows():
    sig = 'sig' if r['significant'] \
        else 'ns'
    print(f'    {r["protein"]:<15} '
           f'logFC={r["logFC"]:+.3f} '
           f'FDR={r["adj_p_value"]:.2e} '
           f'{sig}')

de_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_de_v2.csv',
    index=False)
print('  DE saved')

# ── RF ────────────────────────────────
print('\nRandom Forest...')
y_rf = (labels_consensus ==
        hyper_label).astype(int)

rf   = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
    class_weight='balanced')
cv   = StratifiedKFold(
    n_splits=5, shuffle=True,
    random_state=SEED)
aucs = []

for fold,(tr,te) in enumerate(
        cv.split(X_int, y_rf)):
    rf.fit(X_int[tr], y_rf[tr])
    prob = rf.predict_proba(
        X_int[te])[:,1]
    auc  = roc_auc_score(
        y_rf[te], prob)
    aucs.append(auc)
    print(f'  Fold {fold+1}: '
           f'AUC={auc:.4f}')

mean_auc = np.mean(aucs)
std_auc  = np.std(aucs)
print(f'\n  CV AUC = {mean_auc:.4f} '
       f'+- {std_auc:.4f}')

rf.fit(X_int, y_rf)
rf_df = pd.DataFrame({
    'protein'  : prot_cols,
    'gini_imp' : rf.feature_importances_,
}).sort_values(
    'gini_imp',
    ascending=False
).reset_index(drop=True)

de_sig_set = set(
    de_df[de_df['significant']][
        'protein'].tolist())
top20 = set(
    rf_df.head(20)['protein'].tolist())
top30 = set(
    rf_df.head(min(30,len(rf_df)))[
        'protein'].tolist())
overlap20 = top20 & de_sig_set
overlap30 = top30 & de_sig_set

print(f'  RF-DE top20: '
       f'{len(overlap20)}/20 '
       f'({len(overlap20)/20*100:.0f}%)')

rf_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_rf_v2.csv',
    index=False)
print('  RF saved')

# ── GMM ───────────────────────────────
print('\nGMM validation...')
Z_best = np.load(
    f'{DATA_DIR}/'
    'ibdome_latent_dae_dec.npy')
labels_gmm = GaussianMixture(
    n_components=2,
    random_state=SEED,
    n_init=10
).fit_predict(Z_best)
ari = adjusted_rand_score(
    labels_consensus, labels_gmm)
print(f'  GMM ARI: {ari:.4f}')

# ── Chemokine x Vascular ──────────────
print('\nChemokine x Vascular...')
CHEMOKINE_LIST = [
    'IL8','MCP-3','MCP-1','CXCL11',
    'CXCL9','CXCL1','CCL4','CCL19',
    'CXCL5','CCL3','CXCL6','CXCL10',
    'CCL28','CCL25','CCL20']
VASCULAR_LIST  = [
    'VEGFA','HGF','FGF-21','FGF-19',
    'FGF-5','LIF','ARTN','NRTN',
    'GDNF','FGF-23','Beta-NGF']
CYTOKINE_LIST  = [
    'IL6','IL-17C','IL-17A','OSM',
    'IL18','IL10','TNF','IFN-gamma']

def pathway_score(X, plist, avail):
    present = [p for p in plist
                if p in avail]
    if not present:
        return np.zeros(X.shape[0]),[]
    idx = [avail.index(p)
            for p in present]
    return X[:,idx].mean(axis=1), present

sc_chemo, used_chemo = pathway_score(
    X_int, CHEMOKINE_LIST, prot_cols)
sc_vasc,  used_vasc  = pathway_score(
    X_int, VASCULAR_LIST, prot_cols)
sc_cyto,  used_cyto  = pathway_score(
    X_int, CYTOKINE_LIST, prot_cols)

r_cv, p_cv = stats.pearsonr(
    sc_chemo, sc_vasc)
r_cc, p_cc = stats.pearsonr(
    sc_chemo, sc_cyto)

print(f'  Chemo proteins  : '
       f'{len(used_chemo)} {used_chemo}')
print(f'  Vascular prots  : '
       f'{len(used_vasc)} {used_vasc}')
print(f'  Chemo x Vasc    : '
       f'r={r_cv:.4f} p={p_cv:.4e}')
print(f'  UKB reference   : r=0.749')
print(f'  Delta           : '
       f'{abs(r_cv-0.749):.4f}')

# ── Update metadata ───────────────────
meta['cluster']   = labels_consensus
meta['stability'] = stab_orig
meta['subtype']   = [
    'Hyperinflammatory'
    if l==hyper_label else 'Quiescent'
    for l in labels_consensus]

meta.to_csv(
    f'{DATA_DIR}/'
    'ibdome_cd_final_v2.csv',
    index=False)

# Save validation
with open(
        f'{DATA_DIR}/'
        'ibdome_cd_validation_v2.json',
        'w') as f:
    json.dump({
        'cohort'                  : 'IBDome_CD',
        'chemokine_vascular_r'    : round(r_cv,5),
        'chemokine_cytokine_r'    : round(r_cc,5),
        'ukb_reference_r'         : 0.749,
        'delta_r'                 : round(abs(r_cv-0.749),5),
        'n_chemo_used'            : len(used_chemo),
        'n_vasc_used'             : len(used_vasc),
        'chemokine_proteins_used' : used_chemo,
        'vascular_proteins_used'  : used_vasc,
    }, f, indent=2)

# ── Summary ───────────────────────────
print('\nFINAL SUMMARY')
print('='*55)
summary = {
    'Cohort'         : 'IBDome_CD',
    'Disease'        : "Crohn's Disease",
    'Country'        : 'Germany',
    'N_patients'     : N,
    'N_proteins'     : P,
    'k'              : 2,
    'Silhouette'     : round(sil_final,4),
    'Stability'      : round(stab_mean,4),
    'N_unstable'     : int(n_unstable),
    'GMM_ARI'        : round(ari,4),
    'N_hyperinflam'  : int(
        cluster_sizes[hyper_label]),
    'N_quiescent'    : int(
        cluster_sizes[quiet_label]),
    'DE_sig'         : int(n_sig),
    'DE_up_hyper'    : int(n_up_c2),
    'DE_up_quiet'    : int(n_up_c1),
    'RF_AUC'         : round(mean_auc,4),
    'RF_AUC_std'     : round(std_auc,4),
    'RF_DE_top20'    : len(overlap20),
    'RF_DE_top30'    : len(overlap30),
    'Chemo_Vasc_r'   : round(r_cv,4),
    'UKB_ref_r'      : 0.749,
    'Delta_r'        : round(
        abs(r_cv-0.749),4),
}
for k,v in summary.items():
    print(f'  {k:<20} {v}')

pd.DataFrame([summary]).to_csv(
    f'{TABLES_DIR}/'
    'ibdome_cd_summary_v2.csv',
    index=False)

print('\nAll files saved')
print('\nExpected results:')
print('  DE sig      : 58')
print('  RF AUC      : 0.990')
print('  Hyperinflam : 137')
print('  Quiescent   : 64')
print('  Chemo r     : 0.735')

IBDome CD — Original Setup
X shape     : (201, 61)
Labels      : [ 64 137]
N match     : True

Clustering:
  Silhouette : 0.8342
  Stability  : 0.8857
  Unstable   : 0
  Sizes      : [64, 137]

Inflammatory scores:
  C0 (n=64): -0.6584
  C1 (n=137): 0.3076
  Hyperinflam: C2 (n=137)
  HGF C0=-0.9067 C1=0.4236
  HGF higher in C2

Differential abundance...
  Significant : 58
  Higher C2   : 58
  Higher C1   : 0

  Top 10:
    HGF             logFC=+1.330 FDR=3.86e-21 sig
    CXCL11          logFC=+1.254 FDR=2.26e-18 sig
    TNFRSF9         logFC=+1.171 FDR=6.02e-16 sig
    IL18            logFC=+1.196 FDR=1.75e-15 sig
    SLAMF1          logFC=+1.140 FDR=2.18e-15 sig
    CCL20           logFC=+1.151 FDR=7.90e-15 sig
    CXCL10          logFC=+1.117 FDR=8.42e-15 sig
    CCL3            logFC=+1.139 FDR=1.06e-14 sig
    IL-18R1         logFC=+1.074 FDR=2.51e-14 sig
    CXCL1           logFC=+1.137 FDR=2.86e-14 sig
  DE saved

Random Forest...
  Fold 1: AUC=1.0000
  Fold 2: AUC=0.9821
  Fold